In [ ]:
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd()
PRETRAINED_WEIGHTS = BASE_DIR / "best.pt"
DATA_YAML = Path(r"D:\Tabmed\dataset_v5_balanced\data.yaml")

print(
    f"Базовые веса: {PRETRAINED_WEIGHTS} (найден: {PRETRAINED_WEIGHTS.is_file()})"
)
print(f"Датасет: {DATA_YAML} (найден: {DATA_YAML.is_file()})")

model = YOLO(str(PRETRAINED_WEIGHTS))

results = model.train(
    data=str(DATA_YAML),
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,  # RTX 4060 Ti
    workers=2,  # Стабильно для Windows DataLoader
    lr0=0.002,  # Пониженный LR, чтобы бережно дообучить на новых ракурсах
    lrf=0.01,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    project=str(BASE_DIR / "runs" / "detect"),
    name="train_merged_v5",
    exist_ok=True,
)

print(
    "\nОбучение завершено! Новые веса сохранены в runs/detect/train_merged_v5/weights/best.pt"
)

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

RUN_DIR = Path(r"D:\Tabmed\Tabmed_ml\runs\detect\train_merged_v5")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Confusion Matrix
cm_path = RUN_DIR / "confusion_matrix.png"
if cm_path.is_file():
    cm_img = cv2.cvtColor(cv2.imread(str(cm_path)), cv2.COLOR_BGR2RGB)
    axes[0].imshow(cm_img)
    axes[0].set_title("Матрица ошибок (Confusion Matrix)", fontsize=13)
    axes[0].axis("off")

# 2. Результаты обучения по эпохам
res_path = RUN_DIR / "results.png"
if res_path.is_file():
    res_img = cv2.cvtColor(cv2.imread(str(res_path)), cv2.COLOR_BGR2RGB)
    axes[1].imshow(res_img)
    axes[1].set_title("Динамика потерь и mAP (Results)", fontsize=13)
    axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Новые веса train_merged_v5
WEIGHTS_V5 = Path(
    r"D:\Tabmed\Tabmed_ml\runs\detect\train_merged_v5\weights\best.pt"
)
model = YOLO(str(WEIGHTS_V5))

DIR_D1 = Path(r"D:\Tabmed\dataset1\images")
BASE_DIR = Path(r"D:\Tabmed\Tabmed_ml")

# Список контрольных файлов для проверки
test_files = [
    ("Базовый test.jpg", BASE_DIR / "test.jpg"),
    ("Ладонь (проверка на пальцы)", DIR_D1 / "20261021_232806.jpg"),
    ("Капсулы (вытянутая форма)", DIR_D1 / "IMG_20260202_111543.jpg"),
    ("Таблетки на столе", DIR_D1 / "IMG_20260202_111049.jpg"),
]

# Фильтруем только существующие файлы
valid_tests = [(label, p) for label, p in test_files if p.is_file()]

# Если какого-то файла нет по имени, дополним первыми фото из dataset1
if len(valid_tests) < 4:
    extra = list(DIR_D1.glob("*.jpg"))[: 4 - len(valid_tests)]
    for p in extra:
        valid_tests.append((f"Dataset1: {p.name}", p))

fig, axes = plt.subplots(2, 2, figsize=(15, 14))
axes = axes.flatten()

print(f"{'Тест':<28} | {'Capsules':<9} | {'Tablets':<8} | {'Детали детекции'}")
print("-" * 75)

for i, (label, img_path) in enumerate(valid_tests[:4]):
    res = model.predict(source=str(img_path), conf=0.30, verbose=False)[0]

    # Рендер разметки с увеличенными линиями
    annotated = res.plot(line_width=3, font_size=1.2)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    # Безопасное масштабирование превью для исключения OOM в VS Code
    h, w = annotated_rgb.shape[:2]
    scale = 600 / max(h, w)
    preview = cv2.resize(
        annotated_rgb,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA,
    )

    counts = {"capsule": 0, "tablet": 0}
    details = []
    for b in res.boxes:
        c_name = res.names[int(b.cls[0].item())]
        conf = float(b.conf[0].item())
        counts[c_name] = counts.get(c_name, 0) + 1
        details.append(f"{c_name} ({conf:.0%})")

    summary_str = f"caps: {counts['capsule']}, tab: {counts['tablet']}"
    det_str = ", ".join(details) if details else "ничего не найдено"

    print(f"{label:<28} | {counts['capsule']:<9} | {counts['tablet']:<8} | {det_str}")

    axes[i].imshow(preview)
    axes[i].set_title(f"{label}\n[{summary_str}]", fontsize=11)
    axes[i].axis("off")

plt.tight_layout()
plt.show()